# OpenArm 2.0 Phase 1 — Dataset Exploration & Quality Audit

This Colab notebook audits a LeRobot-format dataset with both teleoperation data and a wrist/egocentric camera stream.

Recommended dataset:

```python
REPO_ID = "lerobot/libero_object_image"
```

The notebook covers:

- episode count and trajectory length distribution
- state/action/timestamp key inspection
- joint-state NaN/Inf checks
- timestamp gap checks
- joint velocity spike checks
- wrist-camera blur / exposure / frozen-frame checks
- sample visualization
- CSV/JSON report export

The main design idea is to treat robot state/action and wrist video as synchronized modalities and preserve episode/timestamp indices for later curation.


In [ ]:
# Colab setup
!pip -q install -U lerobot huggingface_hub opencv-python-headless matplotlib pandas tqdm pyyaml


In [ ]:
import os
import json
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

# Robust import for different LeRobot versions
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
except Exception:
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

# Dataset with both robot state/action and wrist camera stream
REPO_ID = "lerobot/libero_object_image"

OUTPUT_DIR = Path("/content/openarm_phase1_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this small for the first run.
# After it works, increase to None to scan all episodes.
MAX_EPISODES_TO_AUDIT = 20

# Teleoperation thresholds
MIN_EPISODE_LEN = 20
MAX_JOINT_VELOCITY = 20.0
MAX_TIMESTAMP_GAP_RATIO = 3.0

# Egocentric / wrist video thresholds
# These are intentionally conservative starter values.
# Calibrate after inspecting score distributions.
BLUR_THRESHOLD = 30.0
EXPOSURE_THRESHOLD = 0.50
FROZEN_FRAME_THRESHOLD = 0.5

# Only sample some frames for speed
VIDEO_FRAME_STRIDE = 5

print("Output directory:", OUTPUT_DIR)
print("Dataset:", REPO_ID)


## 1. Load dataset and inspect sample keys


In [ ]:
dataset = LeRobotDataset(REPO_ID)

print("Loaded dataset:", REPO_ID)
print("Number of frames:", len(dataset))

sample = dataset[0]

print("\nSample keys:")
for k in sample.keys():
    v = sample[k]
    if isinstance(v, torch.Tensor):
        print(f"  {k}: tensor shape={tuple(v.shape)}, dtype={v.dtype}")
    else:
        print(f"  {k}: type={type(v)}")

if hasattr(dataset, "meta"):
    print("\nDataset has meta object.")
    if hasattr(dataset.meta, "features"):
        print("Features:")
        print(dataset.meta.features)


## 2. Utility functions for robust key detection


In [ ]:
def to_numpy(x):
    """Convert torch tensor / scalar / list / PIL-like object to numpy."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    if isinstance(x, np.ndarray):
        return x
    try:
        return np.asarray(x)
    except Exception:
        return x


def scalar(x):
    arr = to_numpy(x)
    if isinstance(arr, np.ndarray):
        if arr.shape == ():
            return arr.item()
        return arr.reshape(-1)[0].item()
    return x


def find_first_key(sample, candidates):
    for key in candidates:
        if key in sample:
            return key
    return None


def find_state_key(sample):
    candidates = [
        "observation.state",
        "observation.joint_positions",
        "observation.joints",
        "state",
    ]
    key = find_first_key(sample, candidates)
    if key is not None:
        return key

    for k in sample.keys():
        kl = k.lower()
        if "state" in kl or "joint" in kl:
            arr = to_numpy(sample[k])
            if isinstance(arr, np.ndarray) and arr.ndim <= 2:
                return k

    return None


def find_action_key(sample):
    candidates = ["action", "actions"]
    key = find_first_key(sample, candidates)
    if key is not None:
        return key

    for k in sample.keys():
        if "action" in k.lower():
            return k

    return None


def find_timestamp_key(sample):
    candidates = ["timestamp", "timestamps", "time"]
    key = find_first_key(sample, candidates)
    if key is not None:
        return key

    for k in sample.keys():
        if "time" in k.lower():
            return k

    return None


def find_image_keys(sample):
    image_keys = []

    for k, v in sample.items():
        kl = k.lower()
        if "image" in kl or "camera" in kl or "frame" in kl:
            arr = to_numpy(v)
            if isinstance(arr, np.ndarray) and arr.ndim >= 3:
                image_keys.append(k)

    return image_keys


def choose_wrist_or_ego_camera(image_keys):
    """Prefer wrist/egocentric camera if present. Otherwise use first image stream."""
    preferred_terms = ["wrist", "ego", "egocentric", "hand"]

    for term in preferred_terms:
        for k in image_keys:
            if term in k.lower():
                return k

    return image_keys[0] if image_keys else None


In [ ]:
sample = dataset[0]

STATE_KEY = find_state_key(sample)
ACTION_KEY = find_action_key(sample)
TIMESTAMP_KEY = find_timestamp_key(sample)
IMAGE_KEYS = find_image_keys(sample)
WRIST_IMAGE_KEY = choose_wrist_or_ego_camera(IMAGE_KEYS)

print("Detected state key:", STATE_KEY)
print("Detected action key:", ACTION_KEY)
print("Detected timestamp key:", TIMESTAMP_KEY)
print("Detected image keys:", IMAGE_KEYS)
print("Selected wrist/egocentric image key:", WRIST_IMAGE_KEY)

if STATE_KEY is None:
    raise ValueError("Could not detect joint/state key. Inspect sample keys above and set STATE_KEY manually.")

if WRIST_IMAGE_KEY is None:
    print("Warning: no image stream detected. Egocentric video audit will be skipped.")


## 3. Group frames by episode


In [ ]:
def get_episode_indices_fast(dataset):
    """Group dataset indices by episode id."""
    episode_to_indices = defaultdict(list)

    # Fast path: LeRobotDataset often exposes hf_dataset
    if hasattr(dataset, "hf_dataset"):
        hfds = dataset.hf_dataset
        if hasattr(hfds, "column_names") and "episode_index" in hfds.column_names:
            episode_indices = hfds["episode_index"]
            for idx, ep in enumerate(episode_indices):
                episode_to_indices[int(ep)].append(idx)
            return dict(episode_to_indices)

    # Fallback path
    for idx in tqdm(range(len(dataset)), desc="Scanning episode_index"):
        sample = dataset[idx]
        if "episode_index" in sample:
            ep = int(scalar(sample["episode_index"]))
        elif "episode_id" in sample:
            ep = int(scalar(sample["episode_id"]))
        else:
            ep = 0
        episode_to_indices[ep].append(idx)

    return dict(episode_to_indices)


episode_to_indices = get_episode_indices_fast(dataset)

print("Number of episodes:", len(episode_to_indices))

episode_lengths = pd.DataFrame([
    {"episode_id": ep, "num_steps": len(indices)}
    for ep, indices in episode_to_indices.items()
]).sort_values("episode_id")

display(episode_lengths.head())
display(episode_lengths["num_steps"].describe())


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(episode_lengths["num_steps"], bins=30)
plt.xlabel("Episode length, frames")
plt.ylabel("Count")
plt.title("Trajectory Length Distribution")
plt.tight_layout()

plot_path = OUTPUT_DIR / "trajectory_length_distribution.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved:", plot_path)


## 4. Teleoperation audit


In [ ]:
def get_episode_arrays(dataset, indices):
    states = []
    actions = []
    timestamps = []

    for local_i, idx in enumerate(indices):
        sample = dataset[idx]

        state = to_numpy(sample[STATE_KEY]).reshape(-1)
        states.append(state)

        if ACTION_KEY is not None:
            action = to_numpy(sample[ACTION_KEY]).reshape(-1)
            actions.append(action)

        if TIMESTAMP_KEY is not None:
            timestamps.append(float(scalar(sample[TIMESTAMP_KEY])))
        else:
            timestamps.append(float(local_i))

    states = np.stack(states, axis=0)
    actions = np.stack(actions, axis=0) if actions else None
    timestamps = np.asarray(timestamps, dtype=np.float64)

    return states, actions, timestamps


def audit_joint_states(states, timestamps):
    report = {}

    report["num_steps"] = int(len(states))
    report["state_dim"] = int(states.shape[1])
    report["has_nan_or_inf"] = bool(not np.isfinite(states).all())

    report["state_min"] = np.nanmin(states, axis=0).tolist()
    report["state_max"] = np.nanmax(states, axis=0).tolist()
    report["state_mean"] = np.nanmean(states, axis=0).tolist()
    report["state_std"] = np.nanstd(states, axis=0).tolist()

    if len(timestamps) >= 2:
        dt = np.diff(timestamps)
        dt_safe = np.maximum(dt, 1e-6)
        median_dt = float(np.median(dt_safe))

        timestamp_gaps = dt_safe > MAX_TIMESTAMP_GAP_RATIO * median_dt
        report["median_dt"] = median_dt
        report["max_dt"] = float(np.max(dt_safe))
        report["num_timestamp_gaps"] = int(timestamp_gaps.sum())

        velocity = np.diff(states, axis=0) / dt_safe[:, None]
        max_abs_vel = np.nanmax(np.abs(velocity), axis=0)

        report["max_abs_joint_velocity_per_dim"] = max_abs_vel.tolist()
        report["max_abs_joint_velocity"] = float(np.nanmax(max_abs_vel))
        report["has_velocity_spike"] = bool(np.nanmax(max_abs_vel) > MAX_JOINT_VELOCITY)
    else:
        report["median_dt"] = None
        report["max_dt"] = None
        report["num_timestamp_gaps"] = 0
        report["max_abs_joint_velocity_per_dim"] = []
        report["max_abs_joint_velocity"] = 0.0
        report["has_velocity_spike"] = False

    return report


def teleop_filter_reasons(ep_len, joint_report):
    reasons = []

    if ep_len < MIN_EPISODE_LEN:
        reasons.append("too_short")

    if joint_report["has_nan_or_inf"]:
        reasons.append("nan_or_inf_joint_state")

    if joint_report["has_velocity_spike"]:
        reasons.append("joint_velocity_spike")

    if joint_report["num_timestamp_gaps"] > 0:
        reasons.append("timestamp_gap")

    return reasons


## 5. Wrist / egocentric video quality audit


In [ ]:
def normalize_frame(frame):
    """Convert tensor/array to uint8 HWC RGB-like image."""
    img = to_numpy(frame)

    if not isinstance(img, np.ndarray):
        img = np.asarray(img)

    # CHW -> HWC
    if img.ndim == 3 and img.shape[0] in [1, 3, 4]:
        img = np.transpose(img, (1, 2, 0))

    # Remove alpha
    if img.ndim == 3 and img.shape[-1] == 4:
        img = img[..., :3]

    # Grayscale -> RGB
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)

    # Convert to uint8
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        if img.max() <= 1.5:
            img = img * 255.0
        img = np.clip(img, 0, 255).astype(np.uint8)

    return img


def blur_score(frame):
    img = normalize_frame(frame)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())


def exposure_score(frame):
    img = normalize_frame(frame)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    dark_ratio = np.mean(gray < 10)
    bright_ratio = np.mean(gray > 245)

    return float(dark_ratio + bright_ratio)


def frame_difference(frame_a, frame_b):
    a = normalize_frame(frame_a).astype(np.float32)
    b = normalize_frame(frame_b).astype(np.float32)

    if a.shape != b.shape:
        b = cv2.resize(b, (a.shape[1], a.shape[0]))

    return float(np.mean(np.abs(a - b)))


def classify_frame_quality(frame, prev_frame=None):
    reasons = []

    b = blur_score(frame)
    e = exposure_score(frame)

    if b < BLUR_THRESHOLD:
        reasons.append("blur")

    if e > EXPOSURE_THRESHOLD:
        reasons.append("bad_exposure")

    diff = None
    if prev_frame is not None:
        diff = frame_difference(prev_frame, frame)
        if diff < FROZEN_FRAME_THRESHOLD:
            reasons.append("frozen_or_duplicate")

    return {
        "valid": len(reasons) == 0,
        "reasons": reasons,
        "blur_score": b,
        "exposure_score": e,
        "frame_difference": diff,
    }


def audit_egocentric_video(dataset, indices, image_key, stride=5):
    if image_key is None:
        return {
            "available": False,
            "image_key": None,
            "num_sampled_frames": 0,
            "num_bad_frames": 0,
            "bad_frame_ratio": None,
            "frame_reports": [],
        }

    frame_reports = []
    prev_frame = None

    sampled_indices = indices[::stride]

    for idx in sampled_indices:
        sample = dataset[idx]
        frame = sample[image_key]

        q = classify_frame_quality(frame, prev_frame)
        q["dataset_index"] = int(idx)
        frame_reports.append(q)

        prev_frame = frame

    num_bad = sum(1 for r in frame_reports if not r["valid"])

    return {
        "available": True,
        "image_key": image_key,
        "num_sampled_frames": len(frame_reports),
        "num_bad_frames": int(num_bad),
        "bad_frame_ratio": float(num_bad / max(1, len(frame_reports))),
        "frame_reports": frame_reports,
    }


## 6. Run Phase 1 audit


In [ ]:
all_episode_ids = sorted(episode_to_indices.keys())

if MAX_EPISODES_TO_AUDIT is not None:
    audit_episode_ids = all_episode_ids[:MAX_EPISODES_TO_AUDIT]
else:
    audit_episode_ids = all_episode_ids

audit_reports = []

for ep_id in tqdm(audit_episode_ids, desc="Auditing episodes"):
    indices = episode_to_indices[ep_id]

    states, actions, timestamps = get_episode_arrays(dataset, indices)

    joint_report = audit_joint_states(states, timestamps)
    video_report = audit_egocentric_video(
        dataset,
        indices,
        WRIST_IMAGE_KEY,
        stride=VIDEO_FRAME_STRIDE,
    )

    reasons = teleop_filter_reasons(len(indices), joint_report)

    # Add video reason at episode level.
    # This is a candidate rejection rule, not a final hard deletion rule.
    if video_report["available"] and video_report["bad_frame_ratio"] is not None:
        if video_report["bad_frame_ratio"] > 0.25:
            reasons.append("high_bad_egocentric_frame_ratio")

    audit_reports.append({
        "episode_id": int(ep_id),
        "num_steps": int(len(indices)),
        "state_key": STATE_KEY,
        "action_key": ACTION_KEY,
        "timestamp_key": TIMESTAMP_KEY,
        "image_keys": IMAGE_KEYS,
        "wrist_image_key": WRIST_IMAGE_KEY,
        "joint_report": joint_report,
        "egocentric_video_report": video_report,
        "keep_candidate": len(reasons) == 0,
        "filter_reasons": reasons,
    })

print("Finished auditing episodes:", len(audit_reports))


## 7. Summarize audit results


In [ ]:
summary_rows = []

for r in audit_reports:
    jr = r["joint_report"]
    vr = r["egocentric_video_report"]

    summary_rows.append({
        "episode_id": r["episode_id"],
        "num_steps": r["num_steps"],
        "keep_candidate": r["keep_candidate"],
        "filter_reasons": ", ".join(r["filter_reasons"]),
        "has_nan_or_inf": jr["has_nan_or_inf"],
        "max_abs_joint_velocity": jr["max_abs_joint_velocity"],
        "num_timestamp_gaps": jr["num_timestamp_gaps"],
        "bad_frame_ratio": vr["bad_frame_ratio"],
        "num_bad_frames": vr["num_bad_frames"],
        "num_sampled_video_frames": vr["num_sampled_frames"],
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)
display(summary_df.describe(include="all"))

csv_path = OUTPUT_DIR / "phase1_audit_summary.csv"
summary_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(summary_df["max_abs_joint_velocity"].dropna(), bins=30)
plt.xlabel("Max absolute joint velocity")
plt.ylabel("Episode count")
plt.title("Joint Velocity Spike Audit")
plt.tight_layout()

path = OUTPUT_DIR / "joint_velocity_audit.png"
plt.savefig(path, dpi=150)
plt.show()

print("Saved:", path)


if "bad_frame_ratio" in summary_df and summary_df["bad_frame_ratio"].notna().any():
    plt.figure(figsize=(7, 4))
    plt.hist(summary_df["bad_frame_ratio"].dropna(), bins=20)
    plt.xlabel("Bad wrist/egocentric frame ratio")
    plt.ylabel("Episode count")
    plt.title("Wrist / Egocentric Video Quality Audit")
    plt.tight_layout()

    path = OUTPUT_DIR / "egocentric_bad_frame_ratio.png"
    plt.savefig(path, dpi=150)
    plt.show()

    print("Saved:", path)


## 8. Debug why video frames were marked bad


In [ ]:
reason_counts = Counter()

for r in audit_reports:
    for fr in r["egocentric_video_report"]["frame_reports"]:
        for reason in fr["reasons"]:
            reason_counts[reason] += 1

print("Video rejection reason counts:")
print(reason_counts)

blur_scores = []
exposure_scores = []
frame_diffs = []

for r in audit_reports:
    for fr in r["egocentric_video_report"]["frame_reports"]:
        blur_scores.append(fr["blur_score"])
        exposure_scores.append(fr["exposure_score"])
        if fr["frame_difference"] is not None:
            frame_diffs.append(fr["frame_difference"])

def print_stats(name, values):
    if len(values) == 0:
        print(name, "no values")
        return
    values = np.asarray(values)
    print(
        f"{name}: min={values.min():.4f}, "
        f"mean={values.mean():.4f}, "
        f"median={np.median(values):.4f}, "
        f"max={values.max():.4f}"
    )

print_stats("Blur score", blur_scores)
print_stats("Exposure score", exposure_scores)
print_stats("Frame diff", frame_diffs)


In [ ]:
if len(blur_scores) > 0:
    plt.figure(figsize=(7, 4))
    plt.hist(blur_scores, bins=30)
    plt.axvline(BLUR_THRESHOLD, linestyle="--", label=f"threshold={BLUR_THRESHOLD}")
    plt.xlabel("Blur score, variance of Laplacian")
    plt.ylabel("Sampled frame count")
    plt.title("Blur Score Distribution")
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / "blur_score_distribution.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)

if len(exposure_scores) > 0:
    plt.figure(figsize=(7, 4))
    plt.hist(exposure_scores, bins=30)
    plt.axvline(EXPOSURE_THRESHOLD, linestyle="--", label=f"threshold={EXPOSURE_THRESHOLD}")
    plt.xlabel("Exposure score, saturated pixel ratio")
    plt.ylabel("Sampled frame count")
    plt.title("Exposure Score Distribution")
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / "exposure_score_distribution.png"
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)


## 9. Visualize image streams


In [ ]:
def show_sample_frames(dataset, episode_to_indices, ep_id, image_key, num_frames=6):
    if image_key is None:
        print("No image key available.")
        return

    indices = episode_to_indices[ep_id]
    chosen = np.linspace(0, len(indices) - 1, num_frames).astype(int)

    plt.figure(figsize=(3 * num_frames, 3))

    for plot_i, local_i in enumerate(chosen):
        idx = indices[local_i]
        sample = dataset[idx]
        frame = normalize_frame(sample[image_key])

        plt.subplot(1, num_frames, plot_i + 1)
        plt.imshow(frame)
        plt.axis("off")
        plt.title(f"idx={idx}")

    plt.suptitle(f"Episode {ep_id}, camera={image_key}")
    plt.tight_layout()
    plt.show()


first_ep = audit_episode_ids[0]
show_sample_frames(dataset, episode_to_indices, first_ep, WRIST_IMAGE_KEY, num_frames=6)


In [ ]:
# Optional: visualize all detected image streams for the first episode.
for image_key in IMAGE_KEYS:
    print("Visualizing:", image_key)
    show_sample_frames(dataset, episode_to_indices, first_ep, image_key, num_frames=6)


## 10. Save JSON report and zip outputs


In [ ]:
def json_safe(obj):
    if isinstance(obj, dict):
        return {k: json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    return obj


report_path = OUTPUT_DIR / "phase1_audit_report.json"

with open(report_path, "w") as f:
    json.dump(json_safe(audit_reports), f, indent=2)

print("Saved full report:", report_path)

# Also save key metadata for README writing.
metadata = {
    "repo_id": REPO_ID,
    "num_frames": len(dataset),
    "num_episodes": len(episode_to_indices),
    "state_key": STATE_KEY,
    "action_key": ACTION_KEY,
    "timestamp_key": TIMESTAMP_KEY,
    "image_keys": IMAGE_KEYS,
    "wrist_image_key": WRIST_IMAGE_KEY,
    "max_episodes_audited": MAX_EPISODES_TO_AUDIT,
    "thresholds": {
        "min_episode_len": MIN_EPISODE_LEN,
        "max_joint_velocity": MAX_JOINT_VELOCITY,
        "max_timestamp_gap_ratio": MAX_TIMESTAMP_GAP_RATIO,
        "blur_threshold": BLUR_THRESHOLD,
        "exposure_threshold": EXPOSURE_THRESHOLD,
        "frozen_frame_threshold": FROZEN_FRAME_THRESHOLD,
        "video_frame_stride": VIDEO_FRAME_STRIDE,
    },
}

metadata_path = OUTPUT_DIR / "phase1_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(json_safe(metadata), f, indent=2)

print("Saved metadata:", metadata_path)


In [ ]:
!zip -r /content/openarm_phase1_outputs.zip /content/openarm_phase1_outputs

print("Download this file from Colab:")
print("/content/openarm_phase1_outputs.zip")


## README interpretation template

After running the notebook, summarize the results like this:

```text
I audited the LeRobot dataset using both teleoperation and wrist-camera streams. 
For the teleoperation modality, I inspected episode length, joint-state validity, timestamp continuity, and velocity spikes. 
For the egocentric modality, I sampled wrist-camera frames and measured blur, exposure saturation, and frozen-frame artifacts.

The main difference between the two modalities is that joint-state filtering is mostly numerical and kinematic, while egocentric filtering is perceptual and threshold-sensitive. 
Therefore, the video filter should be calibrated with score distributions and manual frame inspection before being used to discard full episodes.
```
